# Lab 22-10 · Power Delivery: Interleaving, the Package Resonance and Damping

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, optimize
import pdn


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Four interleaved phases

*Predict* one phase's ripple and the four-phase sum for 12 V to 0.8 V, 100 nH, 1 MHz.

In [2]:
d = 0.8 / 12
print(f"one phase {pdn.phase_ripple(12, d, 100e-9, 1e6):.2f} A, four phases {pdn.summed_ripple(4, 12, d, 100e-9, 1e6):.2f} A")
s = ngspice("multiphase.cir")
print({kk: s[kk] for kk in ("ipp1", "ippsum")})

one phase 7.47 A, four phases 5.87 A
{'ipp1': 7.461412, 'ippsum': 5.844572}


## Part 2 · Phase count

*Predict* which N cancel the ripple completely at D = 1/15.

In [3]:
n = np.arange(1, 31)
rel = [pdn.summed_ripple(k, 12, d, 100e-9, 1e6) / pdn.phase_ripple(12, d, 100e-9, 1e6) for k in n]
plt.figure(); plt.plot(n, rel, "o-"); plt.xlabel("phases N"); plt.ylabel("summed / single ripple"); plt.show()
print("complete cancellation at N =", [int(k) for k, r in zip(n, rel) if r < 1e-9])

complete cancellation at N = [15, 30]


## Part 3 · The package resonance

*Predict* f0, Z0, the impedance peak (about Z0^2/R_ESR) and the first droop for Example 22-10.6.

In [4]:
L, C, R = 2e-12, 20e-6, 0.1e-3
f0, z0 = pdn.resonance(L, C)
print(f"f0 {f0/1e6:.1f} MHz, Z0 {z0*1e6:.0f} uohm, peak estimate {z0**2/R*1e3:.2f} mohm, undamped droop {pdn.first_droop_undamped(100, L, C)*1e3:.1f} mV")
s = ngspice("pdn.cir")
print({kk: s[kk] for kk in ("zpk", "fpk", "droop")})

f0 25.2 MHz, Z0 316 uohm, peak estimate 1.00 mohm, undamped droop 31.6 mV


{'zpk': 0.001048861, 'fpk': 25263880.0, 'droop': 0.0265392}


## Part 4 · How much damping?

*Predict*: does more ESR always reduce the first droop?

In [5]:
def droop(r):
    def f(t, y):
        il, vc = y
        return [-(vc + r * (il - 100)) / L, (il - 100) / C]
    sol = integrate.solve_ivp(f, (0, 100e-9), [0, 0], max_step=2e-12, rtol=1e-9, atol=1e-14)
    il, vc = sol.y
    return -(vc + r * (il - 100)).min()


rs = np.linspace(0, 0.5e-3, 26)
ds = [droop(r) for r in rs]
plt.figure(); plt.plot(rs * 1e3, np.array(ds) * 1e3); plt.xlabel("ESR (mohm)"); plt.ylabel("first droop (mV)"); plt.show()
best = optimize.minimize_scalar(droop, bounds=(1e-6, 0.6e-3), method="bounded")
print(f"minimum droop {best.fun*1e3:.1f} mV at ESR {best.x*1e3:.2f} mohm (Z0/2 = {z0/2*1e3:.2f} mohm)")

minimum droop 25.6 mV at ESR 0.17 mohm (Z0/2 = 0.16 mohm)


## Part 5 · Distribution voltage

*Predict* the busbar loss of Example 22-10.2 at 48, 400 and 800 V for a 120 kW rack.

In [6]:
r_loop = 2 * pdn.bar_resistance(1.5, 600e-6)
for vv in (48, 400, 800):
    print(f"{vv} V: {120e3/vv:.0f} A, distributed loss {pdn.busbar_loss(120e3/vv, r_loop):.2f} W")

48 V: 2500 A, distributed loss 179.17 W
400 V: 300 A, distributed loss 2.58 W
800 V: 150 A, distributed loss 0.65 W


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.